# Next-Horizon Evaluation: Template-Adversarial Representation Learning (DANN)
## Experiments 3 & 4: DANN-LiLT and DANN-BROS on VRDU Registration Forms

**IE 643 Course Project** — Cross-Template Key-Value Field Extraction from Scanned Forms  
**Authors:** Team Unemployed and Unsupervised (Param Mehta 23b2439, Arjoe Basak 23b1295) · IIT Bombay  

---

### Context & Progress Summary
In the previous run on VRDU Registration Forms ($N=200$ matched sample size):
- **Experiment 1 (BROS Baseline):** MTL 22.70% -> UTL 7.48% (67.0% relative drop). Demonstrates that naive 512-token truncation on multi-page forms drops 83.3% of document content.
- **Experiment 2 (BROS + 384-token Sliding Window + Coordinate Jitter):** MTL **92.36%** -> UTL **71.27%** (**22.8% relative drop**).
  - Slashes relative generalization collapse from 29.8% (LiLT) down to **22.8%**.
  - Increases absolute unseen template accuracy from 64.24% to **71.27% (+7.03 percentage points)**.

**Objective of This Notebook:**
Run **Experiment 3 (DANN-LiLT)** and **Experiment 4 (DANN-BROS)** in **lightweight evaluation mode** (`--no-save-weights`).
By keeping model weights strictly in memory during validation/test evaluation and saving only `result.json` / `metrics.json`, disk usage stays under **10 MB** (preventing Kaggle 20GB disk exhaustion).

--- 
## 0 — Environment Setup, Dual-GPU Check & Repo Checkout

In [ ]:
import os, subprocess, sys, shutil

GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
    print("[+] Successfully retrieved GITHUB_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[*] Notice: GITHUB_TOKEN not found in Kaggle Secrets ({e}). Defaulting to public HTTPS.")

REPO_NAME = "ABasakEE/KeyValue-Extraction"
REPO_DIR = "/kaggle/working/KeyValue-Extraction"
BRANCH_NAME = "feat/bros-relative-layout"

if not os.path.exists(REPO_DIR):
    print(f"[*] Cloning {REPO_NAME} (branch: {BRANCH_NAME})...")
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_NAME}.git" if GITHUB_TOKEN else f"https://github.com/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "-b", BRANCH_NAME, clone_url, REPO_DIR], check=True)
else:
    print(f"[*] Pulling latest commits on {BRANCH_NAME}...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH_NAME], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.chdir(REPO_DIR)
print(f"[+] Current working directory: {os.getcwd()}")

# Check available GPU devices
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv

--- 
## 1 — Install Dependencies & Verify Unit Tests

In [ ]:
!pip install -q seqeval python-Levenshtein pyyaml

# Run unit tests for BROS and DANN modules
!python tests/test_bros_pipeline.py
!python tests/test_dann.py

--- 
## 2 — Clone & Verify Google VRDU Dataset

In [ ]:
import os, subprocess

vrdu_dir = "data/vrdu"
if not os.path.exists(os.path.join(vrdu_dir, "registration-form", "main", "dataset.jsonl.gz")):
    print("[*] Cloning Google VRDU dataset from https://github.com/google-research-datasets/vrdu.git...")
    os.makedirs("data", exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/google-research-datasets/vrdu.git", vrdu_dir], check=True)
    print("[+] VRDU dataset cloned successfully.")
else:
    print("[+] VRDU dataset already present.")

# Verify registration-form subcorpus
reg_p = os.path.join(vrdu_dir, "registration-form", "main", "dataset.jsonl.gz")
print(f"Checking registration-form: exists = {os.path.exists(reg_p)}")

--- 
## 3 — Disk Space Management & Safety Verification
Ensures at least 10 GB of free space exists on `/kaggle/working`. If residual 500MB checkpoints exist from previous runs, this cell removes heavy `.pt` files while preserving all `.json` metrics.

In [ ]:
import os, glob

# Check free disk space
!df -h /kaggle/working

# Prune any residual heavy best_model.pt files to guarantee clean execution
residual_pts = glob.glob("results/**/*.pt", recursive=True)
if residual_pts:
    print(f"[*] Found {len(residual_pts)} residual .pt checkpoint(s). Pruning to reclaim disk space...")
    for pt in residual_pts:
        try:
            os.remove(pt)
        except Exception:
            pass
    print("[+] Pruned heavy checkpoints. Preserved all metrics.json and result.json files.")

!df -h /kaggle/working

---
## 5 — Experiment 3: Template-Adversarial Representation Learning (DANN-LiLT)
Applies Gradient Reversal Layer and Template Discriminator to LiLT to mathematically penalize template-specific spatial coordinates.

**Dual-GPU Seed Parallelism:** Automatically spawns 2 concurrent workers across `cuda:0` and `cuda:1`. GPU 0 trains Seed 13 while GPU 1 trains Seed 21/42 concurrently.

In [ ]:
# Run DANN-LiLT across both GPUs (dual-GPU seed parallelism is automatic)
# Note: To evaluate exactly 2 seeds (e.g. GPU 0 = Seed 13, GPU 1 = Seed 42), append: --seeds 13,42
!python src/train.py --config configs/vrdu_dann_lilt.yaml --protocol matched --no-save-weights

---
## 6 — Experiment 4: Unified DANN-BROS (Relative Layout + Adversarial Invariance)
The ultimate architectural synthesis: pairing BROS 2D relative spatial attention with template-adversarial regularization.

**Dual-GPU Seed Parallelism:** Dispatches tasks across `cuda:0` and `cuda:1` in parallel.

In [ ]:
# Run DANN-BROS across both GPUs in parallel
# Note: To evaluate exactly 2 seeds (e.g. GPU 0 = Seed 13, GPU 1 = Seed 42), append: --seeds 13,42
!python src/train.py --config configs/vrdu_dann_bros.yaml --protocol matched --no-save-weights

--- 
## 6 — Experiment 4: Unified DANN-BROS (Relative Layout + Adversarial Invariance)
The ultimate architectural synthesis: pairing BROS 2D relative spatial attention with template-adversarial regularization.

In [ ]:
# Run DANN-BROS with --no-save-weights
!python src/train.py --config configs/vrdu_dann_bros.yaml --protocol matched --no-save-weights

--- 
## 7 — Master Comparative Benchmark & Generalization Gap Report
Aggregates all 6 experimental horizons under matched sample size ($N=200$):

In [ ]:
import os, json, glob
from pathlib import Path
import pandas as pd

def evaluate_matched_results(results_dir: str):
    mtl_files = glob.glob(f"{results_dir}/MTL_fold*/result.json")
    utl_files = glob.glob(f"{results_dir}/UTL_fold*/result.json")
    
    def get_mean(files):
        scores = []
        for f in files:
            try:
                data = json.loads(Path(f).read_text(encoding='utf-8'))
                scores.append(data.get('test', {}).get('f1', 0.0))
            except Exception:
                pass
        return sum(scores) / len(scores) if scores else None
    
    mtl_f1 = get_mean(mtl_files)
    utl_f1 = get_mean(utl_files)
    if mtl_f1 is not None and utl_f1 is not None:
        gap = mtl_f1 - utl_f1
        rel_drop = (gap / mtl_f1 * 100) if mtl_f1 > 0 else 0.0
        return mtl_f1, utl_f1, gap, rel_drop
    return None

# Master benchmark registry (including completed previous runs)
benchmark_data = [
    {
        "Architecture / Method": "1. Baseline LiLT (Absolute Coordinates)",
        "MTL (Seen) F1": 0.9156,
        "UTL (Unseen) F1": 0.6424,
        "Gap (MTL - UTL)": 0.2731,
        "Relative Drop (%)": 29.8,
        "Status": "Benchmark Reference"
    },
    {
        "Architecture / Method": "2. Standalone RSA-GNN (Geometric Graph)",
        "MTL (Seen) F1": 0.8597,
        "UTL (Unseen) F1": 0.4205,
        "Gap (MTL - UTL)": 0.4392,
        "Relative Drop (%)": 51.1,
        "Status": "Benchmark Reference"
    },
    {
        "Architecture / Method": "3. BROS Baseline (Pairwise Relative Spatial PE)",
        "MTL (Seen) F1": 0.2270,
        "UTL (Unseen) F1": 0.0748,
        "Gap (MTL - UTL)": 0.1522,
        "Relative Drop (%)": 67.0,
        "Status": "Completed (Exp 1)"
    },
    {
        "Architecture / Method": "4. BROS + 384-tok Window + Coordinate Jitter",
        "MTL (Seen) F1": 0.9236,
        "UTL (Unseen) F1": 0.7127,
        "Gap (MTL - UTL)": 0.2110,
        "Relative Drop (%)": 22.8,
        "Status": "Completed (Exp 2)"
    },
]

# Check DANN-LiLT (Exp 3)
res_dann_lilt = evaluate_matched_results("results/vrdu_dann_lilt")
if res_dann_lilt:
    m, u, g, r = res_dann_lilt
    benchmark_data.append({
        "Architecture / Method": "5. DANN-LiLT (Template-Adversarial Regularization)",
        "MTL (Seen) F1": round(m, 4),
        "UTL (Unseen) F1": round(u, 4),
        "Gap (MTL - UTL)": round(g, 4),
        "Relative Drop (%)": round(r, 1),
        "Status": "Completed (Exp 3)"
    })
else:
    benchmark_data.append({
        "Architecture / Method": "5. DANN-LiLT (Template-Adversarial Regularization)",
        "MTL (Seen) F1": None, "UTL (Unseen) F1": None, "Gap (MTL - UTL)": None, "Relative Drop (%)": None,
        "Status": "Pending Run"
    })

# Check DANN-BROS (Exp 4)
res_dann_bros = evaluate_matched_results("results/vrdu_dann_bros")
if res_dann_bros:
    m, u, g, r = res_dann_bros
    benchmark_data.append({
        "Architecture / Method": "6. DANN-BROS (Relative Layout + Adversarial Invariance)",
        "MTL (Seen) F1": round(m, 4),
        "UTL (Unseen) F1": round(u, 4),
        "Gap (MTL - UTL)": round(g, 4),
        "Relative Drop (%)": round(r, 1),
        "Status": "Completed (Exp 4)"
    })
else:
    benchmark_data.append({
        "Architecture / Method": "6. DANN-BROS (Relative Layout + Adversarial Invariance)",
        "MTL (Seen) F1": None, "UTL (Unseen) F1": None, "Gap (MTL - UTL)": None, "Relative Drop (%)": None,
        "Status": "Pending Run"
    })

df = pd.DataFrame(benchmark_data)
print("\n" + "="*90)
print("MASTER CROSS-TEMPLATE GENERALIZATION BENCHMARK (N=200 MATCHED PROTOCOL)")
print("="*90)
print(df.to_string(index=False))

# Save combined summary
os.makedirs("results", exist_ok=True)
df.to_csv("results/master_architectural_benchmark.csv", index=False)
print("\n[+] Master benchmark saved to results/master_architectural_benchmark.csv")